# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
# Step 0 - Import required libraries
# pandas and numpy for data manipulation
# scikit-learn for preprocessing, modeling, and evaluation

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
# Step 0.1 - Load the Spaceship Titanic dataset
# The dataset is loaded directly from the provided GitHub URL

spaceship = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv"
)

# Display the first rows to understand the data structure
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# Step 1 – Feature Engineering, Cleaning & Scaling
# Step 1.1 - Drop rows with missing target values
# Transported is the target variable we want to predict

spaceship = spaceship.dropna(subset=["Transported"])

In [4]:
# Step 1.2 - Drop columns that are identifiers or not useful for modeling

spaceship = spaceship.drop(columns=["PassengerId", "Name"])

In [5]:
# Step 1.3 - Convert boolean target to numeric
# True -> 1, False -> 0

spaceship["Transported"] = spaceship["Transported"].astype(int)

In [6]:
# Step 1.4 - Handle missing values
# Numerical columns: fill with median
# Categorical columns: fill with mode

num_cols = spaceship.select_dtypes(include=["int64", "float64"]).columns
cat_cols = spaceship.select_dtypes(include="object").columns

for col in num_cols:
    spaceship[col] = spaceship[col].fillna(spaceship[col].median())

for col in cat_cols:
    spaceship[col] = spaceship[col].fillna(spaceship[col].mode()[0])

C:\Users\famil\AppData\Local\Temp\ipykernel_13648\508413387.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spaceship[col] = spaceship[col].fillna(spaceship[col].mode()[0])


In [7]:
# Step 1.5 - Encode categorical variables using one-hot encoding

spaceship = pd.get_dummies(spaceship, drop_first=True)

In [8]:
# Step 1.6 - Separate features and target

X = spaceship.drop(columns="Transported")
y = spaceship["Transported"]

In [9]:
# Step 1.7 - Feature scaling
# StandardScaler is used to normalize numerical features

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

**Perform Train Test Split**

In [10]:
# Step 2 - Split the dataset into training and testing sets
# 80% training data, 20% testing data

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [11]:
# Step 3.1 - Import Bagging Classifier
# Bagging reduces variance by training multiple models on random subsets of data

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [12]:
# Step 3.2 - Initialize Bagging Classifier
# Base estimator: Decision Tree
# bootstrap=True means sampling with replacement (Bagging)

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

In [13]:
# Step 3.3 - Train Bagging model

bagging_model.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                  n_estimators=100, n_jobs=-1, random_state=42)

In [14]:
# Step 3.4 - Make predictions

bagging_predictions = bagging_model.predict(X_test)

In [15]:
# Step 3.5 - Evaluate Bagging model

bagging_accuracy = accuracy_score(y_test, bagging_predictions)
bagging_accuracy

0.78205865439908

In [16]:
# Pasting (Sampling WITHOUT replacement)
# Step 3.6 - Initialize Pasting Classifier
# bootstrap=False means sampling without replacement (Pasting)

pasting_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

In [17]:
# Step 3.7 - Train Pasting model

pasting_model.fit(X_train, y_train)

BaggingClassifier(bootstrap=False,
                  estimator=DecisionTreeClassifier(random_state=42),
                  n_estimators=100, n_jobs=-1, random_state=42)

In [18]:
# Step 3.8 - Make predictions

pasting_predictions = pasting_model.predict(X_test)

In [19]:
# Step 3.9 - Evaluate Pasting model

pasting_accuracy = accuracy_score(y_test, pasting_predictions)
pasting_accuracy

0.7659574468085106

- Random Forests

In [20]:
# Step 4.1 - Import Random Forest Classifier
# Random Forest combines Bagging + feature randomness

from sklearn.ensemble import RandomForestClassifier

In [21]:
# Step 4.2 - Initialize Random Forest model
# n_estimators = number of trees

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

In [22]:
# Step 4.3 - Train the Random Forest model

rf_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [23]:
# Make Predictions
# Step 4.4 - Predict using Random Forest

rf_predictions = rf_model.predict(X_test)

In [24]:
# Step 4.5 - Evaluate Random Forest model

rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_accuracy

0.78953421506613

- Gradient Boosting

In [25]:
# Step 5.1 - Import Gradient Boosting model

from sklearn.ensemble import GradientBoostingClassifier

In [26]:
# Step 5.2 - Initialize the Gradient Boosting model

gb_model = GradientBoostingClassifier(random_state=42)

In [27]:
# Step 5.3 - Train the model

gb_model.fit(X_train, y_train)

GradientBoostingClassifier(random_state=42)

In [28]:
# Step 5.4 - Make predictions on the test set

gb_predictions = gb_model.predict(X_test)

In [29]:
# Step 5.5 - Evaluate model performance using accuracy

gb_accuracy = accuracy_score(y_test, gb_predictions)
gb_accuracy

0.7809085681426107

- Adaptive Boosting

In [30]:
# Step 6.1 - Import AdaBoost classifier

from sklearn.ensemble import AdaBoostClassifier

In [31]:
# Step 6.2 - Initialize the AdaBoost model

ada_model = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

In [32]:
# Step 6.3 - Train the AdaBoost model

ada_model.fit(X_train, y_train)

AdaBoostClassifier(n_estimators=100, random_state=42)

In [33]:
# Step 6.4 - Make predictions on the test set

ada_predictions = ada_model.predict(X_test)

In [34]:
# Step 6.5 - Evaluate model performance

ada_accuracy = accuracy_score(y_test, ada_predictions)
ada_accuracy

0.7711328349626222

In [35]:
# Final Model Comparison
# Step 7 - Compare all ensemble model accuracies

print("Gradient Boosting Accuracy:", gb_accuracy)
print("AdaBoost Accuracy:", ada_accuracy)
print("Bagging Accuracy:", bagging_accuracy)
print("Pasting Accuracy:", pasting_accuracy)
print("Random Forest Accuracy:", rf_accuracy)

Gradient Boosting Accuracy: 0.7809085681426107
AdaBoost Accuracy: 0.7711328349626222
Bagging Accuracy: 0.78205865439908
Pasting Accuracy: 0.7659574468085106
Random Forest Accuracy: 0.78953421506613


Which model is the best and why?

### Final Model Selection

Among all ensemble methods tested, **Random Forest** achieved the best performance.

Reasons:
- It combines bagging with random feature selection, reducing overfitting
- It is robust to noise and works well with high-dimensional data
- It captures complex interactions better than single-tree models

Gradient Boosting also performed strongly, but Random Forest provided a better balance
between bias and variance for this dataset.